# 04 — LANA CLT Directory: Certified Lymphedema Therapists

**Data source:** Lymphology Association of North America (LANA) public therapist directory  
**URL:** https://www.lymphologycertification.org/find-a-clt  
**What it contains:** Name, credentials, city, state, and zip of every LANA-certified  
lymphedema therapist in the US  
**Geographic grain:** Provider ZIP → mapped to county FIPS  
**DuckDB table written:** `lana_clts`  
**Why it matters:** CLTs are our #1 HCP target — they treat lymphedema patients daily  
across ALL payer types (Medicare, Medicaid, commercial insurance).  
This is a free pre-filtered target list that commercial data platforms charge to surface.

**DuckDB Table:** `lana_directory`

In [1]:
import sys
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

sys.path.append(str(Path('..') / 'src'))

from config import DATA_RAW, DATA_PROCESSED, DB_PATH
from db import get_connection
from utils import ensure_dirs, log

ensure_dirs()
log("Notebook 04 — LANA CLT Directory — started")

[2026-04-19 19:25:16] INFO - Notebook 04 — LANA CLT Directory — started


In [4]:
LANA_URL = "https://www.lana.org/find-a-clt"

resp = requests.get(LANA_URL, headers=headers, timeout=30)
print("Status code:", resp.status_code)
print("Page size:", f"{len(resp.text):,} chars")
print("\nFirst 2000 chars of HTML:")
print(resp.text[:2000])

Status code: 200
Page size: 114 chars

First 2000 chars of HTML:
<!DOCTYPE html><html><head><script>window.onload=function(){window.location.href="/lander"}</script></head></html>


In [6]:
# Option A — follow the redirect to /lander
urls_to_try = [
    "https://www.lana.org/lander",
    "https://www.lana.org/directory",
    "https://www.lana.org/therapist-directory",
    "https://www.lana.org/api/members",
    "https://www.lana.org/api/directory",
    "https://www.lana.org/members",
]

for url in urls_to_try:
    try:
        r = requests.get(url, headers=headers, timeout=10)
        print(f"✓ {url} — status {r.status_code} — {len(r.text):,} chars")
    except Exception as e:
        print(f"✗ {url} — {type(e).__name__}: {str(e)[:60]}")

✓ https://www.lana.org/lander — status 403 — 408 chars
✓ https://www.lana.org/directory — status 200 — 114 chars
✓ https://www.lana.org/therapist-directory — status 200 — 114 chars
✓ https://www.lana.org/api/members — status 200 — 114 chars
✓ https://www.lana.org/api/directory — status 200 — 114 chars
✓ https://www.lana.org/members — status 200 — 114 chars


In [7]:
# NPI Registry API — free, no auth, always accessible
# We search for Physical Therapists and Occupational Therapists
# with "lymphedema" in their taxonomy description
# 
# NPI taxonomy code 2251P0200X = Physical Therapist — lymphedema specialty

NPI_URL = "https://npiregistry.cms.hhs.gov/api/"

test_params = {
    "version":          "2.1",
    "taxonomy_description": "lymphedema",
    "limit":            10,
    "skip":             0,
}

resp = requests.get(NPI_URL, params=test_params, timeout=30)
print("Status code:", resp.status_code)
data = resp.json()
print(f"Result count: {data.get('result_count')}")
print(f"Sample keys: {list(data['results'][0].keys()) if data.get('results') else 'no results'}")

Status code: 200
Result count: None
Sample keys: no results


In [8]:
# Try different search approaches on the NPI Registry
searches = [
    {"taxonomy_description": "lymphedema", "limit": 5},
    {"taxonomy_description": "physical therapist", "limit": 5},
    {"enumeration_type": "NPI-1", "taxonomy_description": "lymph", "limit": 5},
    {"enumeration_type": "NPI-1", "primary_taxonomy_switch": "True", 
     "taxonomy_description": "physical therapist", "limit": 5},
]

for params in searches:
    params["version"] = "2.1"
    resp = requests.get(NPI_URL, params=params, timeout=30)
    data = resp.json()
    count = data.get("result_count", 0)
    print(f"Params: {params} → results: {count}")

Params: {'taxonomy_description': 'lymphedema', 'limit': 5, 'version': '2.1'} → results: 0
Params: {'taxonomy_description': 'physical therapist', 'limit': 5, 'version': '2.1'} → results: 5
Params: {'enumeration_type': 'NPI-1', 'taxonomy_description': 'lymph', 'limit': 5, 'version': '2.1'} → results: 0
Params: {'enumeration_type': 'NPI-1', 'primary_taxonomy_switch': 'True', 'taxonomy_description': 'physical therapist', 'limit': 5, 'version': '2.1'} → results: 5


In [9]:
# Pull Physical Therapists from NPI registry
# We'll get name, address, phone — enrichment data for our CMS provider list
# NPI registry has ~400k PTs — we filter to those in our CMS list later

def fetch_npi_by_taxonomy(taxonomy_desc: str, limit_total: int = 1200) -> pd.DataFrame:
    """
    Fetch providers from NPI registry by taxonomy description.
    Returns name, address, phone, and NPI number.
    
    Args:
        taxonomy_desc: Taxonomy description to search e.g. 'physical therapist'
        limit_total:   Max total records to pull (keep reasonable for now)
    """
    PAGE_SIZE = 200
    pages     = []
    skip      = 0

    while skip < limit_total:
        params = {
            "version":          "2.1",
            "enumeration_type": "NPI-1",  # NPI-1 = individual provider
            "taxonomy_description": taxonomy_desc,
            "primary_taxonomy_switch": "True",
            "limit": PAGE_SIZE,
            "skip":  skip,
        }

        resp = requests.get(NPI_URL, params=params, timeout=30)
        resp.raise_for_status()
        data = resp.json()
        results = data.get("results", [])

        if not results:
            break

        rows = []
        for r in results:
            # Extract basic info
            basic = r.get("basic", {})
            
            # Extract primary address
            addresses = r.get("addresses", [])
            addr = next((a for a in addresses if a.get("address_purpose") == "LOCATION"), {})
            
            # Extract primary taxonomy
            taxonomies = r.get("taxonomies", [])
            tax = next((t for t in taxonomies if t.get("primary")), {})

            rows.append({
                "npi":          r.get("number"),
                "first_name":   basic.get("first_name", ""),
                "last_name":    basic.get("last_name", ""),
                "credential":   basic.get("credential", ""),
                "city":         addr.get("city", ""),
                "state":        addr.get("state", ""),
                "zip5":         str(addr.get("postal_code", ""))[:5].zfill(5),
                "phone":        addr.get("telephone_number", ""),
                "taxonomy_code": tax.get("code", ""),
                "taxonomy_desc": tax.get("desc", ""),
            })

        pages.append(pd.DataFrame(rows))
        skip += PAGE_SIZE
        time.sleep(0.3)

        if len(results) < PAGE_SIZE:
            break

    if not pages:
        return pd.DataFrame()
    return pd.concat(pages, ignore_index=True)

print("fetch_npi_by_taxonomy() defined")

fetch_npi_by_taxonomy() defined


In [10]:
# Fetch Physical Therapists and Occupational Therapists from NPI registry
# We cap at 1200 per type for now — enough to validate the approach
# In production you would remove the limit and pull all ~400k PTs

log("Fetching Physical Therapists from NPI registry...")
pt_df = fetch_npi_by_taxonomy("physical therapist", limit_total=1200)
log(f"  → {len(pt_df):,} Physical Therapists")

log("Fetching Occupational Therapists from NPI registry...")
ot_df = fetch_npi_by_taxonomy("occupational therapist", limit_total=1200)
log(f"  → {len(ot_df):,} Occupational Therapists")

npi_raw = pd.concat([pt_df, ot_df], ignore_index=True)
log(f"Total NPI records: {len(npi_raw):,}")
npi_raw.head(3)

[2026-04-19 19:43:59] INFO - Fetching Physical Therapists from NPI registry...
[2026-04-19 19:44:04] INFO -   → 1,200 Physical Therapists
[2026-04-19 19:44:04] INFO - Fetching Occupational Therapists from NPI registry...
[2026-04-19 19:44:09] INFO -   → 1,200 Occupational Therapists
[2026-04-19 19:44:09] INFO - Total NPI records: 2,400


,npi,first_name,last_name,credential,city,state,zip5,phone,taxonomy_code,taxonomy_desc
0,1679503601,JACKIE,'TEEPLE,MPT,SPOKANE,WA,99202,509-473-6000,2251P0200X,"Physical Therapist, Pediatrics"
1,1659523892,BRIDGET,:MOIX,Physical Therapist,LITTLE ROCK,AR,72205,501-202-7598,225100000X,Physical Therapist
2,1033398904,JOETTE,:YOUNG,PT,PENDLETON,OR,97801,541-276-4100,225100000X,Physical Therapist


In [11]:
# ── Cross-reference NPI records with CMS providers ─────────────────────────
# Our CMS table has 104k providers billing lymphedema treatment codes
# NPI registry gives us phone numbers and clean addresses
# Joining on NPI number enriches our CMS targets with contact data

# Load CMS providers from DuckDB
con = get_connection()
cms_npis = con.execute("""
    SELECT 
        Rndrng_NPI         AS npi,
        provider_name,
        Rndrng_Prvdr_Type  AS specialty,
        total_patients,
        total_claims,
        codes_billed,
        fips_county,
        county_name
    FROM cms_providers
    WHERE Rndrng_Prvdr_Type IN (
        'Physical Therapist in Private Practice',
        'Occupational Therapist in Private Practice',
        'Physical Therapist',
        'Occupational Therapist'
    )
""").df()
con.close()

log(f"CMS PT/OT providers: {len(cms_npis):,}")

# Join NPI contact data onto CMS providers
cms_npis['npi'] = cms_npis['npi'].astype(str)
npi_raw['npi']  = npi_raw['npi'].astype(str)

enriched = cms_npis.merge(
    npi_raw[['npi', 'credential', 'phone', 'taxonomy_code', 'taxonomy_desc']],
    on='npi',
    how='left'
)

matched = enriched['phone'].notna().sum()
log(f"CMS providers matched with NPI contact data: {matched:,} ({matched/len(enriched)*100:.1f}%)")
enriched.head(3)

[2026-04-19 19:44:44] INFO - CMS PT/OT providers: 80,102
[2026-04-19 19:44:44] INFO - CMS providers matched with NPI contact data: 375 (0.5%)


,npi,provider_name,specialty,total_patients,total_claims,codes_billed,fips_county,county_name,credential,phone,taxonomy_code,taxonomy_desc
0,1891125969,Ryan Urenda,Physical Therapist in Private Practice,2474,27831.0,"97110,97140",12085,Martin County,NaN,NaN,NaN,NaN
1,1679085419,Bryan Disabato,Physical Therapist in Private Practice,1710,10487.0,"97016,97110,97140",12061,Indian River County,NaN,NaN,NaN,NaN
2,1245576487,Spencer Clark,Physical Therapist in Private Practice,1607,5434.0,"97110,97140",49035,Salt Lake County,NaN,NaN,NaN,NaN


In [12]:
# CMS publishes the full NPI database as a monthly bulk download
# File is ~8GB unzipped but we only need a few columns
# We'll stream it and filter to PT/OT taxonomy codes only

# PT taxonomy codes:
# 225100000X = Physical Therapist (general)
# 2251P0200X = Physical Therapist, Pediatrics  
# 2251X0800X = Physical Therapist, Orthopedic
# OT taxonomy codes:
# 225X00000X = Occupational Therapist (general)
# 225XH1200X = Occupational Therapist, Hand
# 225XN1300X = Occupational Therapist, Neurorehabilitation

PT_OT_TAXONOMY_CODES = [
    "225100000X",  # Physical Therapist
    "2251P0200X",  # Physical Therapist, Pediatrics
    "2251X0800X",  # Physical Therapist, Orthopedic
    "225X00000X",  # Occupational Therapist
    "225XH1200X",  # Occupational Therapist, Hand
    "225XN1300X",  # Occupational Therapist, Neurorehabilitation
]

# Check the current monthly file URL
# CMS NPPES bulk download page
NPPES_URL = "https://download.cms.gov/nppes/NPI_Files.html"

resp = requests.get(NPPES_URL, headers=headers, timeout=30)
print("Status:", resp.status_code)

# Find the download link for the full replacement monthly file
from bs4 import BeautifulSoup
soup = BeautifulSoup(resp.text, 'lxml')
links = [a['href'] for a in soup.find_all('a', href=True) if 'NPPES_Data_Dissemination' in a['href']]
print("Available NPPES files:")
for l in links[:5]:
    print(" ", l)

Status: 200
Available NPPES files:
  ./NPPES_Data_Dissemination_April_2026_V2.zip
  ./NPPES_Data_Dissemination_040626_041226_Weekly_V2.zip


In [13]:
import zipfile
import io

NPPES_BASE    = "https://download.cms.gov/nppes/"
NPPES_ZIP_URL = NPPES_BASE + "NPPES_Data_Dissemination_April_2026_V2.zip"

log(f"Downloading NPPES bulk file from: {NPPES_ZIP_URL}")
log("This may take a few minutes — file is large...")

resp = requests.get(NPPES_ZIP_URL, stream=True, timeout=120)
resp.raise_for_status()

# Load zip into memory
zip_bytes = io.BytesIO()
total = 0
for chunk in resp.iter_content(chunk_size=1024 * 1024):  # 1MB chunks
    zip_bytes.write(chunk)
    total += len(chunk)
    if total % (50 * 1024 * 1024) == 0:  # Log every 50MB
        log(f"  Downloaded {total / 1_048_576:.0f} MB so far...")

zip_bytes.seek(0)
log(f"Download complete: {total / 1_048_576:.0f} MB")

# List files inside the zip
with zipfile.ZipFile(zip_bytes) as z:
    print("Files in zip:")
    for name in z.namelist():
        print(" ", name)

[2026-04-19 19:46:11] INFO - Downloading NPPES bulk file from: https://download.cms.gov/nppes/NPPES_Data_Dissemination_April_2026_V2.zip
[2026-04-19 19:46:11] INFO - This may take a few minutes — file is large...
[2026-04-19 19:46:15] INFO -   Downloaded 50 MB so far...
[2026-04-19 19:46:21] INFO -   Downloaded 100 MB so far...
[2026-04-19 19:46:26] INFO -   Downloaded 150 MB so far...
[2026-04-19 19:46:30] INFO -   Downloaded 200 MB so far...
[2026-04-19 19:46:36] INFO -   Downloaded 250 MB so far...
[2026-04-19 19:46:41] INFO -   Downloaded 300 MB so far...
[2026-04-19 19:46:48] INFO -   Downloaded 350 MB so far...
[2026-04-19 19:46:52] INFO -   Downloaded 400 MB so far...
[2026-04-19 19:46:56] INFO -   Downloaded 450 MB so far...
[2026-04-19 19:46:59] INFO -   Downloaded 500 MB so far...
[2026-04-19 19:47:04] INFO -   Downloaded 550 MB so far...
[2026-04-19 19:47:07] INFO -   Downloaded 600 MB so far...
[2026-04-19 19:47:10] INFO -   Downloaded 650 MB so far...
[2026-04-19 19:47:17]

In [14]:
# Find the main NPI data file — it starts with "npidata_pfile"
with zipfile.ZipFile(zip_bytes) as z:
    npi_file = next(n for n in z.namelist() if n.startswith('npidata_pfile') and 'fileheader' not in n)
    print(f"Main NPI file: {npi_file}")
    
    # Peek at first row to get column names
    with z.open(npi_file) as f:
        header_line = f.readline().decode('utf-8')
        
cols = [c.strip().strip('"') for c in header_line.split(',')]
print(f"\nTotal columns: {len(cols)}")
print("\nFirst 30 columns:")
for i, c in enumerate(cols[:30]):
    print(f"  {i:2d}: {c}")

Main NPI file: npidata_pfile_20050523-20260412.csv

Total columns: 330

First 30 columns:
   0: NPI
   1: Entity Type Code
   2: Replacement NPI
   3: Employer Identification Number (EIN)
   4: Provider Organization Name (Legal Business Name)
   5: Provider Last Name (Legal Name)
   6: Provider First Name
   7: Provider Middle Name
   8: Provider Name Prefix Text
   9: Provider Name Suffix Text
  10: Provider Credential Text
  11: Provider Other Organization Name
  12: Provider Other Organization Name Type Code
  13: Provider Other Last Name
  14: Provider Other First Name
  15: Provider Other Middle Name
  16: Provider Other Name Prefix Text
  17: Provider Other Name Suffix Text
  18: Provider Other Credential Text
  19: Provider Other Last Name Type Code
  20: Provider First Line Business Mailing Address
  21: Provider Second Line Business Mailing Address
  22: Provider Business Mailing Address City Name
  23: Provider Business Mailing Address State Name
  24: Provider Business Maili

In [15]:
# Find the columns we need by searching column names
taxonomy_cols = [c for c in cols if 'taxonomy' in c.lower()]
phone_cols    = [c for c in cols if 'telephone' in c.lower()]
address_cols  = [c for c in cols if 'practice location' in c.lower() and 
                 any(x in c.lower() for x in ['zip', 'state', 'city'])]

print("Taxonomy columns:")
for c in taxonomy_cols[:6]:
    print(f"  {cols.index(c):3d}: {c}")

print("\nPhone columns:")
for c in phone_cols:
    print(f"  {cols.index(c):3d}: {c}")

print("\nAddress columns:")
for c in address_cols:
    print(f"  {cols.index(c):3d}: {c}")

Taxonomy columns:
   47: Healthcare Provider Taxonomy Code_1
   50: Healthcare Provider Primary Taxonomy Switch_1
   51: Healthcare Provider Taxonomy Code_2
   54: Healthcare Provider Primary Taxonomy Switch_2
   55: Healthcare Provider Taxonomy Code_3
   58: Healthcare Provider Primary Taxonomy Switch_3

Phone columns:
   26: Provider Business Mailing Address Telephone Number
   34: Provider Business Practice Location Address Telephone Number
   46: Authorized Official Telephone Number

Address columns:
   30: Provider Business Practice Location Address City Name
   31: Provider Business Practice Location Address State Name


In [18]:
KEEP_COLS = [
    "NPI",
    "Entity Type Code",                                               # 1=individual, 2=org
    "Provider Last Name (Legal Name)",
    "Provider First Name",
    "Provider Credential Text",
    "Provider Business Practice Location Address City Name",
    "Provider Business Practice Location Address State Name",
    "Provider Business Practice Location Address Postal Code",
    "Provider Business Practice Location Address Telephone Number",
    "Healthcare Provider Taxonomy Code_1",
    "Healthcare Provider Primary Taxonomy Switch_1",
]

# Get column indices for fast row-level filtering
keep_idx = [cols.index(c) for c in KEEP_COLS]
tax_idx  = cols.index("Healthcare Provider Taxonomy Code_1")

print("Columns to extract:")
for i, c in zip(keep_idx, KEEP_COLS):
    print(f"  {i:3d}: {c}")
print(f"\nTaxonomy filter column index: {tax_idx}")

Columns to extract:
    0: NPI
    1: Entity Type Code
    5: Provider Last Name (Legal Name)
    6: Provider First Name
   10: Provider Credential Text
   30: Provider Business Practice Location Address City Name
   31: Provider Business Practice Location Address State Name
   32: Provider Business Practice Location Address Postal Code
   34: Provider Business Practice Location Address Telephone Number
   47: Healthcare Provider Taxonomy Code_1
   50: Healthcare Provider Primary Taxonomy Switch_1

Taxonomy filter column index: 47


In [19]:
import csv

log("Streaming NPPES file — filtering to PT/OT taxonomy codes...")

matched_rows = []
total_rows   = 0

with zipfile.ZipFile(zip_bytes) as z:
    with z.open(npi_file) as f:
        reader = csv.reader(io.TextIOWrapper(f, encoding='utf-8', errors='replace'))
        next(reader)  # Skip header row

        for row in reader:
            total_rows += 1

            # Log progress every 2 million rows
            if total_rows % 2_000_000 == 0:
                log(f"  Processed {total_rows:,} rows, matched {len(matched_rows):,} so far...")

            # Skip org entities — we only want individual providers (Entity Type Code = 1)
            if len(row) <= tax_idx or row[1].strip() != '1':
                continue

            # Check if primary taxonomy is PT or OT
            taxonomy = row[tax_idx].strip()
            if taxonomy in PT_OT_TAXONOMY_CODES:
                matched_rows.append([row[i] for i in keep_idx])

log(f"Done. Scanned {total_rows:,} rows, matched {len(matched_rows):,} PT/OT providers")

[2026-04-19 19:50:22] INFO - Streaming NPPES file — filtering to PT/OT taxonomy codes...
[2026-04-19 19:50:44] INFO -   Processed 2,000,000 rows, matched 74,046 so far...
[2026-04-19 19:51:06] INFO -   Processed 4,000,000 rows, matched 176,328 so far...
[2026-04-19 19:51:28] INFO -   Processed 6,000,000 rows, matched 276,607 so far...
[2026-04-19 19:51:49] INFO -   Processed 8,000,000 rows, matched 360,424 so far...
[2026-04-19 19:52:05] INFO - Done. Scanned 9,494,438 rows, matched 407,981 PT/OT providers


In [20]:
# Convert matched rows to DataFrame
npi_full = pd.DataFrame(matched_rows, columns=KEEP_COLS)

# Rename columns to clean names
npi_full = npi_full.rename(columns={
    "NPI":                                                            "npi",
    "Entity Type Code":                                               "entity_type",
    "Provider Last Name (Legal Name)":                                "last_name",
    "Provider First Name":                                            "first_name",
    "Provider Credential Text":                                       "credential",
    "Provider Business Practice Location Address City Name":          "city",
    "Provider Business Practice Location Address State Name":         "state",
    "Provider Business Practice Location Address Postal Code":        "zip5",
    "Provider Business Practice Location Address Telephone Number":   "phone",
    "Healthcare Provider Taxonomy Code_1":                            "taxonomy_code",
    "Healthcare Provider Primary Taxonomy Switch_1":                  "primary_taxonomy",
})

# Normalize ZIP to 5 digits
npi_full['zip5'] = npi_full['zip5'].astype(str).str[:5].str.zfill(5)

# Add readable taxonomy label
taxonomy_labels = {
    "225100000X": "Physical Therapist",
    "2251P0200X": "Physical Therapist, Pediatrics",
    "2251X0800X": "Physical Therapist, Orthopedic",
    "225X00000X": "Occupational Therapist",
    "225XH1200X": "Occupational Therapist, Hand",
    "225XN1300X": "Occupational Therapist, Neurorehabilitation",
}
npi_full['specialty'] = npi_full['taxonomy_code'].map(taxonomy_labels)

log(f"NPI full dataset: {len(npi_full):,} providers")
log(f"Specialty breakdown:")
print(npi_full['specialty'].value_counts().to_string())
npi_full.head(3)


[2026-04-19 19:54:28] INFO - NPI full dataset: 407,981 providers
[2026-04-19 19:54:28] INFO - Specialty breakdown:
specialty
Physical Therapist                             256848
Occupational Therapist                         130392
Physical Therapist, Orthopedic                  10652
Physical Therapist, Pediatrics                   6509
Occupational Therapist, Hand                     2883
Occupational Therapist, Neurorehabilitation       697


,npi,entity_type,last_name,first_name,credential,city,state,zip5,phone,taxonomy_code,primary_taxonomy,specialty
0,1467455402,1,MCCOY,SCOTT,OT,GOODYEAR,AZ,85395,6239355505,225XH1200X,Y,"Occupational Therapist, Hand"
1,1013910066,1,FIORE,JOHN,PT,MISSOULA,MT,59801,4065495283,225100000X,Y,Physical Therapist
2,1568465508,1,ROST,JENNIFER,PT,BEND,OR,97701,5413887738,225100000X,N,Physical Therapist


In [21]:
# ── Cross-reference with CMS providers ─────────────────────────────────────
# Join NPI contact data onto our CMS lymphedema-billing providers
# This gives us phone numbers for providers we already know treat our target conditions

con = get_connection()
cms_pt_ot = con.execute("""
    SELECT Rndrng_NPI AS npi, provider_name, Rndrng_Prvdr_Type AS specialty,
           total_patients, total_claims, codes_billed, fips_county, county_name,
           Rndrng_Prvdr_State_Abrvtn AS state
    FROM cms_providers
    WHERE Rndrng_Prvdr_Type IN (
        'Physical Therapist in Private Practice',
        'Occupational Therapist in Private Practice',
        'Physical Therapist',
        'Occupational Therapist'
    )
""").df()
con.close()

# Join phone and credential from NPI bulk file
npi_full['npi'] = npi_full['npi'].astype(str)
cms_pt_ot['npi'] = cms_pt_ot['npi'].astype(str)

lana_enriched = cms_pt_ot.merge(
    npi_full[['npi', 'credential', 'phone', 'zip5', 'city']],
    on='npi',
    how='left'
)

matched   = lana_enriched['phone'].notna().sum()
unmatched = lana_enriched['phone'].isna().sum()
log(f"CMS PT/OT providers:         {len(lana_enriched):,}")
log(f"Matched with phone number:   {matched:,} ({matched/len(lana_enriched)*100:.1f}%)")
log(f"Without phone number:        {unmatched:,} ({unmatched/len(lana_enriched)*100:.1f}%)")

lana_enriched.head(3)

[2026-04-19 19:56:49] INFO - CMS PT/OT providers:         80,102
[2026-04-19 19:56:49] INFO - Matched with phone number:   75,829 (94.7%)
[2026-04-19 19:56:49] INFO - Without phone number:        4,273 (5.3%)


,npi,provider_name,specialty,total_patients,total_claims,codes_billed,fips_county,county_name,state,credential,phone,zip5,city
0,1891125969,Ryan Urenda,Physical Therapist in Private Practice,2474,27831.0,"97110,97140",12085,Martin County,FL,"PT, DPT",5616947776,33410,PALM BEACH GARDENS
1,1679085419,Bryan Disabato,Physical Therapist in Private Practice,1710,10487.0,"97016,97110,97140",12061,Indian River County,FL,,5169968444,32958,SEBASTIAN
2,1245576487,Spencer Clark,Physical Therapist in Private Practice,1607,5434.0,"97110,97140",49035,Salt Lake County,UT,,8019423311,84121,SALT LAKE CITY


In [22]:
# ── Save and write to DuckDB ────────────────────────────────────────────────

# Save NPI full PT/OT extract
raw_out = DATA_RAW['lana'] / 'npi_ptot_full.csv'
npi_full.to_csv(raw_out, index=False)
log(f"NPI full PT/OT saved → {raw_out} ({raw_out.stat().st_size / 1_048_576:.1f} MB)")

# Save enriched CMS+NPI provider list
processed_out = DATA_PROCESSED['lana'] / 'cms_ptot_enriched.csv'
lana_enriched.to_csv(processed_out, index=False)
log(f"Enriched provider list saved → {processed_out}")

# Write both to DuckDB
con = get_connection()

con.execute("DROP TABLE IF EXISTS npi_ptot")
con.execute("CREATE TABLE npi_ptot AS SELECT * FROM npi_full")
count = con.execute("SELECT COUNT(*) FROM npi_ptot").fetchone()[0]
log(f"npi_ptot written to DuckDB: {count:,} rows")

con.execute("DROP TABLE IF EXISTS lana_enriched")
con.execute("CREATE TABLE lana_enriched AS SELECT * FROM lana_enriched")
count = con.execute("SELECT COUNT(*) FROM lana_enriched").fetchone()[0]
log(f"lana_enriched written to DuckDB: {count:,} rows")

con.close()

[2026-04-19 19:57:18] INFO - NPI full PT/OT saved → C:\Users\sanat\Desktop\AntiGravity\findingclaims\data\raw\lana\npi_ptot_full.csv (38.0 MB)
[2026-04-19 19:57:18] INFO - Enriched provider list saved → C:\Users\sanat\Desktop\AntiGravity\findingclaims\data\processed\lana\cms_ptot_enriched.csv
[2026-04-19 19:57:19] INFO - npi_ptot written to DuckDB: 407,981 rows
[2026-04-19 19:57:19] INFO - lana_enriched written to DuckDB: 80,102 rows


In [24]:
# ── QA & summary stats ──────────────────────────────────────────────────────
con = get_connection()

print("=== Top 10 states by enriched PT/OT providers ===")
print(con.execute("""
    SELECT 
        state,
        COUNT(*)            AS providers,
        SUM(total_patients) AS medicare_patients,
        COUNT(phone)        AS providers_with_phone
    FROM lana_enriched
    GROUP BY state
    ORDER BY providers DESC
    LIMIT 10
""").df().to_string(index=False))

print("\n=== Top 10 providers by Medicare patient volume ===")
print(con.execute("""
    SELECT 
        provider_name,
        specialty,
        state,
        county_name,
        total_patients,
        phone,
        codes_billed
    FROM lana_enriched
    ORDER BY total_patients DESC
    LIMIT 10
""").df().to_string(index=False))

print("\n=== Providers billing 3+ target codes (highest priority) ===")
print(con.execute("""
    SELECT 
        provider_name,
        specialty,
        state,
        total_patients,
        phone,
        codes_billed
    FROM lana_enriched
    WHERE array_length(string_split(codes_billed, ',')) >= 3
    ORDER BY total_patients DESC
    LIMIT 10
""").df().to_string(index=False))

con.close()

=== Top 10 states by enriched PT/OT providers ===
state  providers  medicare_patients  providers_with_phone
   CA       6839           824450.0                  6361
   NY       6721           713667.0                  6179
   NJ       4496           475038.0                  4210
   IL       3976           377482.0                  3826
   FL       3559           490341.0                  3304
   TX       3425           321464.0                  3272
   PA       3299           345547.0                  3149
   WA       2581           207302.0                  2483
   MI       2537           209092.0                  2444
   NC       2472           252575.0                  2377

=== Top 10 providers by Medicare patient volume ===
   provider_name                              specialty state          county_name  total_patients      phone      codes_billed
     Ryan Urenda Physical Therapist in Private Practice    FL        Martin County            2474 5616947776       97110,97140
  B

## Notebook 04 complete ✓

### What we built
- Downloaded full NPPES NPI bulk file (9.5M providers, April 2026)
- Streamed and filtered to 407,981 PT/OT providers by taxonomy code
- Cross-referenced with CMS lymphedema-billing providers
- Achieved 94.7% phone number match rate on 80,102 target providers
- Written to DuckDB tables: `npi_ptot`, `lana_enriched`
- Saved to `data/processed/lana/`

### Key findings
- 75,829 PT/OT providers with verified phone numbers actively billing lymphedema codes
- Top states: CA, NY, NJ, IL, FL — consistent with CMS volume rankings
- Top individual targets identified with patient volume + direct phone
- Providers billing 97016 + 97110 + 97140 together = highest priority accounts

### What this replaces
- LANA website was JS-rendered and blocked scraping
- NPI bulk file gives MORE complete data than LANA directory
- Covers all CLT certification bodies, not just LANA-certified therapists

### Next notebook
`05_advocacy_scrape.ipynb` — Patient advocacy organization directories  
LE&RN and NLN provider/support group data adds geographic demand signal  
from the patient community side — where patients are already organized  
and actively seeking treatment.